# Blocco B — Costruzione del grafo geopolitico-cyber

**Nodi** = paesi: i 17 del progetto (con i loro 28 profili trimestrali arricchiti = qualitativo dal Blocco A + numerico dai CSV) e i paesi *periferici* esterni che compaiono nelle relazioni.

**Archi** diretti e datati, tutti da dati strutturati (zero LLM):
- **cyber** — attaccante → vittima (CFR)
- **migrazione** — origine → destinazione (UNHCR)
- **coinvolgimento militare** — interventore → teatro (ACLED, forze militari straniere)

La logica di costruzione sta nei moduli `src/graph/build_edges.py` e `build_graph.py`; qui li orchestriamo, controlliamo il risultato e lo visualizziamo.

In [ ]:
import sys
from pathlib import Path
RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE))

from src.graph.build_graph import costruisci_grafo, riepilogo, salva

G = costruisci_grafo()
riepilogo(G)
salva(G)   # -> data/processed/graphs/grafo.pickle + archi.csv

## Controlli di sanità — le relazioni principali

In [ ]:
import pandas as pd
archi = pd.read_csv(RADICE / "data/processed/graphs/archi.csv")

print("TOP attaccanti cyber (n. archi uscenti):")
print(archi[archi.tipo == "cyber"].da.value_counts().head(6), "\n")

print("TOP coinvolgimenti militari (eventi):")
print(archi[archi.tipo == "coinvolgimento_militare"].groupby(["da", "a"]).peso.sum().sort_values(ascending=False).head(6), "\n")

print("TOP flussi migratori (rifugiati):")
print(archi[archi.tipo == "migrazione"].groupby(["da", "a"]).peso.sum().sort_values(ascending=False).head(6))

## Visualizzazione interattiva

**Vista core**: solo i 17 paesi del progetto e le relazioni tra loro — la più leggibile.
Colori archi: 🔴 cyber · 🔵 migrazione · 🟠 militare. Colore nodo = gruppo del paese, dimensione = quante relazioni ha.
Trascina i nodi, zooma, passa il mouse per i dettagli.

In [ ]:
import math, pickle
from collections import defaultdict, Counter
from pyvis.network import Network
from IPython.display import IFrame

GRAPHS = RADICE / "data/processed/graphs"
core = {n for n, d in G.nodes(data=True) if d.get("core")}
COLORI_TIPO = {"cyber": "#e63946", "migrazione": "#457b9d", "coinvolgimento_militare": "#f4a261"}
ETICH_TIPO = {"cyber": "Cyber (attaccante→vittima)", "migrazione": "Migrazione (origine→destinazione)", "coinvolgimento_militare": "Coinvolgimento militare"}
COLORI_GRUPPO = {1: "#d00000", 2: "#f48c06", 3: "#3a86ff", 4: "#2a9d8f"}
NOMI_GRUPPO = {1: "Attori cyber", 2: "Alta instabilità", 3: "Bersagli cyber", 4: "Controllo"}

def aggrega(solo_core=True, soglia_migr=0):
    agg = defaultdict(float)
    for a, b, d in G.edges(data=True):
        if solo_core and not (a in core and b in core):
            continue
        peso = d.get("peso") if d.get("peso") is not None else 1  # cyber non ha peso -> conta 1 incidente
        agg[(a, b, d["tipo"])] += peso
    if soglia_migr:
        agg = {k: v for k, v in agg.items() if not (k[2] == "migrazione" and v < soglia_migr)}
    return agg

def _legenda():
    e = "".join(f'<div><span style="display:inline-block;width:12px;height:12px;background:{c};margin-right:6px;border-radius:2px"></span>{ETICH_TIPO[t]}</div>' for t, c in COLORI_TIPO.items())
    g = "".join(f'<div><span style="display:inline-block;width:12px;height:12px;background:{c};margin-right:6px;border-radius:50%"></span>{NOMI_GRUPPO[k]}</div>' for k, c in COLORI_GRUPPO.items())
    return (f'<div style="position:fixed;top:12px;left:12px;background:#1c1f26cc;color:#eee;padding:12px 14px;'
            f'border-radius:8px;font:13px sans-serif;z-index:999;line-height:1.6">'
            f'<b>Archi</b>{e}<br><b>Nodi (gruppo)</b>{g}'
            f'<div style="margin-top:4px;opacity:.7">◯ tratteggiato = paese periferico</div></div>')

def disegna(agg, nome_file):
    net = Network(height="720px", width="100%", directed=True, bgcolor="#111318", font_color="#eee", cdn_resources="in_line")
    net.barnes_hut(gravity=-9000, spring_length=200, spring_strength=0.02)
    net.set_edge_smooth("dynamic")
    grado = Counter()
    for (a, b, t) in agg:
        grado[a] += 1; grado[b] += 1
    for n in {x for (a, b, t) in agg for x in (a, b)}:
        d = G.nodes[n]; gr = d.get("gruppo")
        if d.get("core"):
            titolo = f"{d.get('nome')} — {NOMI_GRUPPO.get(gr, '?')}\neventi violenti tot 2018-24: {d.get('tot_eventi_violenti')}"
            net.add_node(n, label=n, title=titolo, color=COLORI_GRUPPO.get(gr, "#888"), size=14 + 2.2 * grado[n], borderWidth=2)
        else:
            net.add_node(n, label=n, title=f"{n} (periferico)", color={"background": "#2b2f3a", "border": "#777"},
                         size=10 + grado[n], shapeProperties={"borderDashes": [4, 4]})
    for (a, b, t), p in agg.items():
        net.add_edge(a, b, color=COLORI_TIPO[t], width=1 + math.log1p(p), title=f"{ETICH_TIPO[t]}: {int(p)}", arrows="to")
    path = GRAPHS / nome_file
    net.save_graph(str(path))
    html = path.read_text(encoding="utf-8").replace("<body>", "<body>" + _legenda())
    path.write_text(html, encoding="utf-8")
    return path

disegna(aggrega(solo_core=True), "grafo_core.html")
IFrame(src="../data/processed/graphs/grafo_core.html", width="100%", height=740)

## Vista completa (opzionale)

Include anche i paesi periferici (vittime cyber e teatri esterni). È volutamente **più densa**: i 4 attori cyber (RUS/CHN/PRK/IRN) colpiscono oltre 100 paesi — la "nuvola" di vittime è di per sé un risultato. La migrazione è filtrata a flussi ≥ 50.000 rifugiati per leggibilità. Il grafo completo (172 nodi) è comunque salvato in `grafo.pickle` per l'analisi.

In [ ]:
disegna(aggrega(solo_core=False, soglia_migr=50000), "grafo_completo.html")
IFrame(src="../data/processed/graphs/grafo_completo.html", width="100%", height=740)